# 07 — App layer: vLLM tensor parallel (eugr) + health

**Milestone:** one logical model across **both** GPUs via **TP**; OpenAI-compatible API answers on the head.

**Prerequisites:** complete [`01`](01_first_spark_power_lan_ssh.ipynb)–[`06`](06_nccl_roce_gpu_collectives.ipynb) (or be sure L1–L4+ are green). **Materialize** weights on **each** node under the Hugging Face bind mount — see [`docs/playbook-commands.md`](../docs/playbook-commands.md).

Run **on the head Spark** with Jupyter; not on your laptop.

## Configuration — edit, then **Run All** top-to-bottom

Match `launch-cluster.sh` / NVIDIA playbook choices.

In [ ]:
import os
import subprocess
import sys
import time


def sh(cmd: list[str] | str, *, shell: bool = False) -> subprocess.CompletedProcess:
    if isinstance(cmd, str) and not shell:
        raise ValueError("Pass argv list or use shell=True")
    return subprocess.run(cmd, shell=shell, capture_output=True, text=True)


HEAD_IP = "10.0.0.1"
WORKER_IP = "10.0.0.2"
ETHERNET_IF = "enp1s0f0np0"
IB_IF = "rocep1s0f0"
SSH_USER = "cesarb-ai"
WORKER_SSH = f"{SSH_USER}@{WORKER_IP}"
SPARK_VLLM_DOCKER = "~/spark-vllm-docker"
MODEL_ID = "/root/.cache/huggingface/materialized/Qwen-Qwen2.5-7B-Instruct"
VLLM_CONTAINER_NAME = "vllm_node"

RUN_ZOMBIE_CHECK = True
RUN_LAUNCH_CLUSTER = False
RUN_HEALTH_MONITOR = False

print("Config loaded. Next cells: zombie check → launch command → health.")

## Zombie VRAM (read-only report)

**FlashAttention is the victim, not the suspect** — clear stray processes before blaming kernels.

In [ ]:
def gpu_process_report(label: str, text: str) -> None:
    print(f"\n=== {label} ===")
    print(text.strip() or "(no output)")


if not RUN_ZOMBIE_CHECK:
    print("RUN_ZOMBIE_CHECK is False — skipping")
else:
    r = sh(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"])
    gpu_process_report("Local compute apps", r.stdout + r.stderr)
    r2 = sh(["ssh", WORKER_SSH, "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv"])
    gpu_process_report("Worker compute apps", r2.stdout + r2.stderr)
    r3 = sh(["fuser", "-v", "/dev/nvidia0"])
    print("\n--- fuser /dev/nvidia0 (try sudo if empty) ---")
    print(r3.stdout + r3.stderr or "(no output)")

## Build / run `launch-cluster.sh`

Set **`RUN_LAUNCH_CLUSTER = True`** only after the printed command looks correct. This can run for a long time.

In [ ]:
NODES = f"{HEAD_IP},{WORKER_IP}"
spark_repo = os.path.expanduser(SPARK_VLLM_DOCKER)
launch_cmd = (
    f"cd {spark_repo} && "
    f"./launch-cluster.sh -n {NODES} "
    f"--eth-if {ETHERNET_IF} --ib-if {IB_IF} "
    f"-e MODEL_ID={MODEL_ID} "
    "start"
)
print("Proposed:\n", launch_cmd)
if RUN_LAUNCH_CLUSTER:
    print("\n--- executing ---")
    os.system(launch_cmd)
else:
    print("\n(Set RUN_LAUNCH_CLUSTER = True to execute.)")

## Health: logs, VRAM samples, `/v1/models`

Enable **`RUN_HEALTH_MONITOR`** after the server is up.

In [ ]:
import matplotlib.pyplot as plt
import requests


def smi_used_mib() -> float | None:
    r = sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"])
    if r.returncode != 0:
        return None
    try:
        return float(r.stdout.strip().splitlines()[0].strip())
    except (ValueError, IndexError):
        return None


def remote_smi_used_mib() -> float | None:
    r = sh(
        ["ssh", WORKER_SSH, "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits"]
    )
    if r.returncode != 0:
        return None
    try:
        return float(r.stdout.strip().splitlines()[0].strip())
    except (ValueError, IndexError):
        return None


if not RUN_HEALTH_MONITOR:
    print("RUN_HEALTH_MONITOR is False — skipping")
else:
    rlog = sh(["docker", "logs", "--tail", "20", VLLM_CONTAINER_NAME])
    print("=== docker logs (tail) ===\n", rlog.stdout + rlog.stderr)
    loc, rem = [], []
    for _ in range(8):
        loc.append(smi_used_mib() or 0.0)
        rem.append(remote_smi_used_mib() or 0.0)
        time.sleep(0.75)
    plt.figure(figsize=(8, 3))
    plt.plot(loc, label="head MiB used")
    plt.plot(rem, label="worker MiB used")
    plt.xlabel("sample")
    plt.ylabel("MiB")
    plt.legend()
    plt.title("VRAM (coarse)")
    plt.tight_layout()
    plt.show()
    try:
        resp = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        print("\n=== /v1/models ===\n", resp.text[:2000])
    except requests.RequestException as e:
        print("Smoke failed:", e)

## Next

- Agents: [`examples/langgraph-connection.py`](../examples/langgraph-connection.py)
- One-file rehearsal of **all** gates: [`08_full_stack_console.ipynb`](08_full_stack_console.ipynb)